# Point Cloud to CAD-sequence

In this notebook the complete interactive pipeline for encoding point clouds into a latent space, from which DeepCAD decodes a CAD-sequence.

In [2]:
import os
import sys
import shutil
import glob
import json
import argparse
import importlib

import numpy as np
import pandas as pd
pd.set_option('display.max_rows', None)
import h5py
import torch
import torch.nn.functional as F

from OCC.Core.BRepCheck import BRepCheck_Analyzer
from OCC.Extend.DataExchange import read_step_file, write_step_file
from OCC.Core.STEPControl import STEPControl_Reader
from OCC.Core.StlAPI import StlAPI_Writer
from OCC.Core.BRepMesh import BRepMesh_IncrementalMesh

sys.path.append("..")
sys.path.append("../code")

from dataset import PointCloudEmbeddingSequenceDataset
from models.DeepCAD.config.configAE import ConfigAE
from models.DeepCAD.trainer.trainerAE import TrainerAE
from models.DeepCAD.cadlib.extrude import CADSequence
from models.DeepCAD.cadlib.visualize import vec2CADsolid, create_CAD
from models.DeepCAD.utils.file_utils import ensure_dir

### PointNet++

In [3]:
def inplace_relu(m):
    classname = m.__class__.__name__
    if classname.find('ReLU') != -1:
        m.inplace=True

def load_pointnet():
    sys.path.append(os.path.join('..', 'models','Pointnet_Pointnet2_pytorch', 'models'))
    model = importlib.import_module('pointnet2_cls_ssg')
    classifier = model.get_model(latent_dim, normal_channel=False)
    criterion = model.get_loss_mse()
    classifier.apply(inplace_relu) 
    
    saved_model = torch.load(model_path, map_location=torch.device('cpu'), weights_only=True)
    state_dict = saved_model['model_state_dict']
    if 'module.' in next(iter(state_dict)):
        monitor.log_and_print("Model was saved wrapped in nn.DataParallel.\nRemoving 'module.' from state dict.")
        state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}#
    classifier.eval()
    classifier.load_state_dict(state_dict)
    print(f"Loading PointNet++ from {os.path.abspath(model_path)}")
    return classifier

### DeepCAD

In [4]:
def load_deepcad(cfg):
    tr_agent = TrainerAE(cfg)
    tr_agent.load_ckpt(cfg.ckpt)
    tr_agent.net.eval()
    return tr_agent

### Load data

In [5]:
def get_data(indices, dataset):
    pc_list = []
    lat_rep_list = []  
    pc_paths = []
    cad_seq_list = []
    pc_dir = os.path.join(results_dir, "infered_point_clouds")
    if os.path.exists(pc_dir):
        shutil.rmtree(pc_dir)
    os.mkdir(pc_dir)
    
    for i in indices:
        pc, lat_rep, cad_seq = dataset[i]
        pc_path = os.path.abspath(dataset.get_pc_path(i))
        pc_path_destination = os.path.join(pc_dir, os.path.basename(pc_path))
        shutil.copy2(pc_path, pc_path_destination)
        pc_paths.append(pc_path_destination)
        pc_list.append(pc)
        lat_rep_list.append(lat_rep)
        cad_seq_list.append(cad_seq)

    with h5py.File(h5_file, 'a') as hf:
        dt = h5py.special_dtype(vlen=str)
        path_dataset = hf.create_dataset("pc_paths", shape=(len(pc_paths),), dtype=dt)
        path_dataset[:] = pc_paths
        
    pc_batch = torch.stack(pc_list, dim=0)
    lat_rep_batch = torch.stack(lat_rep_list, dim=0)
    cad_seq_batch = torch.stack(cad_seq_list, dim=0)
    
    return pc_batch, lat_rep_batch, cad_seq_batch

### Inference

In [6]:
def infer_pointnet(indices, dataset, model):
    with h5py.File(h5_file, 'w') as hf:
        z_pred = hf.create_dataset('z_pred', 
                                   shape=(len(indices), latent_dim), 
                                   dtype=np.float32)
        z_target = hf.create_dataset('z_target',
                                     shape=(len(indices), latent_dim),
                                     dtype = np.float32)
        seq_target = hf.create_dataset('seq_target', 
                                       shape=(len(indices), cfg.max_total_len, cfg.n_args + 1), 
                                       dtype=np.int64)
        
        pc, lat_rep, cad_seq = get_data(indices, dataset)
        z_target[:] = lat_rep
        seq_target[:] = cad_seq

        criterion_loader = importlib.import_module('pointnet2_cls_ssg')
        criterion = criterion_loader.get_loss_mse()
        
        with torch.no_grad():
            pc = pc.transpose(2, 1)
            pred, _ = model(pc)
            z_pred[:] = pred.detach()
            loss = criterion(pred, lat_rep)
            print(f"Avg. MSE-Loss: {loss.detach().item():.8f}")
            return pred, cad_seq

In [7]:
def infer_deepcad(pred, cad_seq, tr_agent):
    with h5py.File(h5_file, 'a') as hf:
        seq_pred = hf.create_dataset('seq_pred', 
                                     shape=(pred.shape[0], cfg.max_total_len, cfg.n_args + 1), 
                                     dtype=np.int64)
        cmd_logits = hf.create_dataset('cmd_logits', 
                                       shape=(pred.shape[0], cfg.max_total_len, cfg.n_commands), 
                                       dtype=np.float32)
        args_logits = hf.create_dataset('args_logits', 
                                       shape=(pred.shape[0], cfg.max_total_len, cfg.n_args, cfg.args_dim + 1), 
                                       dtype=np.float32)
        with torch.no_grad():
            pred = pred.unsqueeze(dim = 1)
            output = tr_agent.decode(pred)

            output["tgt_commands"] = cad_seq[:, :, 0] 
            output["tgt_args"] = cad_seq[:, :, 1:]
            loss_dict = tr_agent.loss_func(output)

            cmd_logits[:] = output['command_logits']
            args_logits[:] = output['args_logits']
            
            batch_out_vec = tr_agent.logits2vec(output)
            
            seq_pred[:] = batch_out_vec
            
            print(f"Avg. Command-Loss: {loss_dict['loss_cmd'].detach().cpu().item():.8f}")
            print(f"Avg. Argument-Loss: {loss_dict['loss_args'].detach().cpu().item():.8f}")

### Utils

In [8]:
def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum(axis=-1, keepdims=True)

In [9]:
def cross_entropy(logits, target):
    logits = torch.from_numpy(logits).unsqueeze(0)
    target = torch.tensor([target]).long()
    #print(logits.shape, target.shape)
    return F.cross_entropy(logits, target)

### Visualization

In [10]:
def show_results(idx):
    with h5py.File(h5_file, "r") as hf:
        pc_path = hf['pc_paths'][idx].decode("utf-8")
        args_logits = hf['args_logits'][idx]
        cmd_logits = hf['cmd_logits'][idx]
        seq_pred = hf['seq_pred'][idx]
        seq_target = hf['seq_target'][idx]
        z_pred = hf['z_pred'][idx]
        z_target = hf['z_target'][idx]

    ALL_COMMANDS = ['Line', 'Arc', 'Circle', 'EOS', 'SOL', 'Ext']
    EOS_IDX = ALL_COMMANDS.index('EOS')

    print(f"Point Cloud path: {pc_path}")
    for idx, command in enumerate(ALL_COMMANDS):
        print(f"{idx} -> {command}")

    target_commands = []
    predicted_commands = []
    pred_commands_prob = []
    cmd_loss = []
    cmd_loss_torch = []
    target_commands_prob = []
    
    all_pred_commands = list(seq_pred[:, 0])
    seq_length = list(seq_target[:, 0]).index(EOS_IDX) + 3
    cmd_logits_softmax = softmax(cmd_logits[:seq_length, :])

    for i in range(seq_length):
        predicted_commands.append(int(all_pred_commands[i]))
        target_commands.append(int(seq_target[i, 0]))
        pred_commands_prob.append(round(float(cmd_logits_softmax[i, predicted_commands[i]]) * 100, 5))
        cmd_loss.append(cross_entropy(cmd_logits[i,:], target_commands[i]).item())
        target_commands_prob.append(round(float(cmd_logits_softmax[i, target_commands[i]]) * 100, 5))
    
    df = pd.DataFrame(list(zip(target_commands, predicted_commands, target_commands_prob, pred_commands_prob, cmd_loss)),
                      columns=['trgt', 'pred','prob_trgt', 'prob_pred', 'loss'])
    print(f"Sum CADLoss for {seq_length} commands:  {sum(cmd_loss):.8f}")
    print(f"Mean CADLoss for {seq_length} commands: {np.mean(cmd_loss):.8f}")
    return df

In [11]:
def show_results_args(sample_idx, cmd_idx):
    with h5py.File(h5_file, "r") as hf:
        pc_path = hf['pc_paths'][sample_idx].decode("utf-8")
        args_logits = hf['args_logits'][sample_idx]
        cmd_logits = hf['cmd_logits'][sample_idx]
        seq_pred = hf['seq_pred'][sample_idx]
        seq_target = hf['seq_target'][sample_idx]
        z_pred = hf['z_pred'][sample_idx]
        z_target = hf['z_target'][sample_idx]

    ALL_COMMANDS = ['Line', 'Arc', 'Circle', 'EOS', 'SOL', 'Ext']
    EOS_IDX = ALL_COMMANDS.index('EOS')

    print(f"Point Cloud path: {pc_path}")

    # Extract predicted and target commands
    seq_length = list(seq_target[:, 0]).index(EOS_IDX) + 3
    predicted_commands = seq_pred[:seq_length, 0]      # (60)
    target_commands = seq_target[:seq_length, 0]       # (60)

    # Create lists of predicted and target commands for each of the 16 arguments of each command
    predicted_command_list = [cmd for cmd in predicted_commands[:seq_length] for _ in range(cfg.n_args)] # (16 * seq_length)
    target_command_list = [cmd for cmd in target_commands[:seq_length] for _ in range(cfg.n_args)]       # (16 * seq_length)

    # Extract the arguments logits and calculate softmax
    args_logits = args_logits[:seq_length]        # (seq_length, 16, 257)
    args_softmax = softmax(args_logits)           # (seq_length, 16, 257)

    # Extract the target arguments and softmax probabilities
    target_args = list(seq_target[:seq_length, 1:])                                                           # (seq_length, 16)
    target_cmd_args = torch.tensor([trgt_arg for trgt_cmd_args in target_args for trgt_arg in trgt_cmd_args]) # (16 * seq_length)
    target_cmd_args_sm = args_softmax[torch.repeat_interleave(torch.arange(seq_length), cfg.n_args), torch.arange(cfg.n_args).repeat(seq_length), target_cmd_args + 1] # (16 * seq_length)
    
    # Extract the predicted arguments and softmax probabilities
    predicted_cmd_args = []
    for i, cmd_args in enumerate(args_logits):
        pred_args = np.argmax(cmd_args, -1)
        pred_args = [x - 1 for x in pred_args]
        predicted_cmd_args += list(pred_args)   # (16 * seq_length) 
    predicted_cmd_args_sm = args_softmax[torch.repeat_interleave(torch.arange(seq_length), cfg.n_args), torch.arange(cfg.n_args).repeat(seq_length), [x + 1 for x in predicted_cmd_args]] # (16 * seq_length)
    
    # Calculate cross-entropy loss
    arg_loss = []
    counter = 0
    for i, cmd_args in enumerate(args_logits):
        for j, arg_logits in enumerate(cmd_args):
            loss = cross_entropy(arg_logits, target_cmd_args[counter] + 1)
            arg_loss.append(loss.item())
            counter += 1

    # Extract data for single command view
    if cmd_idx != -1:
        target_command_list = target_command_list[cfg.n_args * cmd_idx:(cfg.n_args * cmd_idx) + cfg.n_args]
        predicted_command_list = predicted_command_list[cfg.n_args * cmd_idx:(cfg.n_args * cmd_idx) + cfg.n_args]
        target_cmd_args = target_cmd_args[cfg.n_args * cmd_idx:(cfg.n_args * cmd_idx) + cfg.n_args]
        predicted_cmd_args = predicted_cmd_args[cfg.n_args * cmd_idx:(cfg.n_args * cmd_idx) + cfg.n_args]
        target_cmd_args_sm = target_cmd_args_sm[cfg.n_args * cmd_idx:(cfg.n_args * cmd_idx) + cfg.n_args]
        predicted_cmd_args_sm = predicted_cmd_args_sm[cfg.n_args * cmd_idx:(cfg.n_args * cmd_idx) + cfg.n_args]
        arg_loss = arg_loss[cfg.n_args * cmd_idx:(cfg.n_args * cmd_idx) + cfg.n_args]

    # Filter out all arguments not involved in the final loss (marked by -1 using a mask in the original code)
    filtered_lists = [[x for x, trgt_arg in zip(lst, target_cmd_args) if trgt_arg != -1] 
                      for lst in [target_command_list, 
                                  predicted_command_list, 
                                  target_cmd_args, 
                                  predicted_cmd_args, 
                                  target_cmd_args_sm, 
                                  predicted_cmd_args_sm, 
                                  arg_loss]]
    target_command_list, predicted_command_list, target_cmd_args, predicted_cmd_args, target_cmd_args_sm, predicted_cmd_args_sm, arg_loss = filtered_lists

    # Create dataframe
    df = pd.DataFrame(list(zip(target_command_list, 
                               predicted_command_list, 
                               [x.item()  for x in target_cmd_args],
                               [x.item()  for x in predicted_cmd_args],
                               [round(x * 100, 5)  for x in target_cmd_args_sm],
                               [round(x * 100, 5) for x in predicted_cmd_args_sm], 
                               [round(x, 5) for x in arg_loss])),
                      columns=['trgt cmd', 'pred cmd', 'trgt', 'pred','prob_trgt', 'prob_pred', 'loss'])
    
    print(f"Mean args loss multiplied by {cfg.loss_weights['loss_args_weight']}: {cfg.loss_weights['loss_args_weight'] * np.mean(arg_loss):.8f}")
    return df

### Export to .step, .stl and .obj

In [12]:
def export2step():
    form = "h5"
    filter = True
    output_dir = os.path.join(results_dir, "step_files")
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    os.mkdir(output_dir)
    h5_path = os.path.join(results_dir, "data.h5")

    with h5py.File(h5_path, 'r') as fp:
        out_vec = fp['seq_pred'][:].astype(np.float64)
        names = fp['pc_paths'][:]
        print(out_vec.shape)
        for i, seq in enumerate(out_vec):
            pc_path = names[i].decode('utf-8')
            out_shape = vec2CADsolid(seq)
    
            if filter:
                analyzer = BRepCheck_Analyzer(out_shape)
                if not analyzer.IsValid():
                    print(f"CAD-sequence of {os.path.basename(pc_path)} is invalid.")
                    continue
    
            pc_name = os.path.splitext(os.path.basename(pc_path))[0]
            save_path = os.path.join(output_dir, pc_name + ".step")
            write_step_file(out_shape, save_path)


In [13]:
def step2stl():
    step_files = glob.glob(os.path.join(results_dir, "step_files", "*.step"))
    obj_dir = os.path.join(results_dir, "stl_files")
    if os.path.exists(obj_dir):
        shutil.rmtree(obj_dir)
    os.mkdir(obj_dir)

    for step_file in step_files:

        save_path = os.path.join(obj_dir, os.path.splitext(os.path.basename(step_file))[0] + ".stl")
    
        step_reader = STEPControl_Reader()
        step_reader.ReadFile(step_file)
        step_reader.TransferRoots()
        shape = step_reader.OneShape()

        BRepMesh_IncrementalMesh(shape, 0.5)

        stl_writer = StlAPI_Writer()
        stl_writer.Write(shape, save_path)

In [14]:
def step2obj(): # .obj files from this function lead to malformed file error in cloud compare
    step_files = glob.glob(os.path.join(results_dir, "step_files", "*.step"))
    obj_dir = os.path.join(results_dir, "obj_files")
    if os.path.exists(obj_dir):
        shutil.rmtree(obj_dir)
    os.mkdir(obj_dir)
    
    for step_file in step_files:
        shape = read_step_file(step_file)
        save_path = os.path.join(obj_dir, os.path.splitext(os.path.basename(step_file))[0] + ".obj")
        write_step_file(shape, save_path)

## Start

### Variables

Store the models in ```experiments```, a results directory will be created for each respective model.

In [15]:
model_name = "best_5"

In [16]:
model_path = os.path.join("experiments", model_name) + ".pth"
results_dir = os.path.join("experiments", model_name + "_results")
if not os.path.exists(results_dir):
    os.mkdir(results_dir)
h5_file = os.path.join(results_dir, "data.h5")
cfg = ConfigAE('test', model_path="../data/latent")
latent_dim = 256

In [17]:
pointnet_plusplus = load_pointnet()
deepcad = load_deepcad(cfg)

Loading PointNet++ from /Users/saidharb/Documents/LocalDocuments/Master-Thesis/Point-Cloud-Reconstruction/notebooks/experiments/best_5.pth
Loading checkpoint from /Users/saidharb/Documents/LocalDocuments/Master-Thesis/Point-Cloud-Reconstruction/data/latent/pretrained/model/ckpt_epoch1000.pth ...


In [18]:
dataset = PointCloudEmbeddingSequenceDataset("../data", 'test')
print(f"Dataset contains {len(dataset)} samples.")

Dataset contains 8038 samples.


In [19]:
indices = [3]

In [20]:
pred, trgt_cad_seq = infer_pointnet(indices, dataset, pointnet_plusplus)

Avg. MSE-Loss: 0.07769267


In [21]:
infer_deepcad(pred, trgt_cad_seq, deepcad)

Avg. Command-Loss: 0.62024128
Avg. Argument-Loss: 7.86331749


In [22]:
show_results(0)

Point Cloud path: experiments/best_5_results/infered_point_clouds/00239323.ply
0 -> Line
1 -> Arc
2 -> Circle
3 -> EOS
4 -> SOL
5 -> Ext
Sum CADLoss for 13 commands:  8.06313781
Mean CADLoss for 13 commands: 0.62024137


,trgt,pred,prob_trgt,prob_pred,loss
0,4,4,99.99925,99.99925,7.510157e-06
1,1,0,0.22564,99.77437,6.094002e+00
2,0,0,99.99993,99.99993,7.152555e-07
3,1,0,14.00296,85.99705,1.965902e+00
4,0,0,99.67797,99.67797,3.225602e-03
5,4,4,100.00000,100.00000,0.000000e+00
6,2,2,99.99995,99.99995,4.768370e-07
7,4,4,100.00000,100.00000,0.000000e+00
8,2,2,100.00000,100.00000,0.000000e+00
9,5,5,100.00000,100.00000,0.000000e+00


In [23]:
show_results_args(0,-1) 

Point Cloud path: experiments/best_5_results/infered_point_clouds/00239323.ply
Mean args loss multiplied by 2.0: 7.86331731


,trgt cmd,pred cmd,trgt,pred,prob_trgt,prob_pred,loss
0,1,0,176,169,9.38706,56.94587,2.36584
1,1,0,128,128,100.00000,100.00000,0.00000
2,1,0,128,128,99.99591,99.99591,0.00004
3,1,0,1,1,83.21034,83.21034,0.18380
4,0,0,176,176,90.10660,90.10660,0.10418
5,0,0,199,223,0.00000,99.72664,17.89966
6,1,0,128,128,100.00000,100.00000,0.00000
7,1,0,199,223,0.00000,99.99502,22.37700
8,1,0,128,128,99.98716,99.98716,0.00013
9,1,0,1,0,35.08765,64.91235,1.04732


In [24]:
export2step()

(1, 60, 17)

*******************************************************************
******        Statistics on Transfer (Write)                 ******

*******************************************************************
******        Transfer Mode = 0  I.E.  As Is       ******
******        Transferring Shape, ShapeType = 2                      ******
** WorkSession : Sending all data
 Step File Name : experiments/best_5_results/step_files/00239323.step(552 ents)  Write  Done


In [25]:
step2stl()

## TODO: 
- export2step and step2stl also for target CAD sequence
- multiple sample visualization for commands and args
- Include quantization to the visualization in the future

### Gedanken

Done: 
- had to finish applications
- first thing I did was refactor training
    - Automatic resume of training if cluster fails
    - Parallelization (30mins/epoch -> 12 mins/epoch)
- implemented test script
- Implemented cosine annealing learning rate -> show new training with better convergence!
- Worked on CAD Loss understanding
- Implemented testing pipeline to make sure the data is alligned
- Finished pc2cad pipeline with thorough understanding of loss for train/val/test
- Created interactive pc2cad
- cmd loss visualization
- args loss visualization

HiWi:
- created SAiL poster
- documented literature research
- made Blensor work

Next:

- train DeepCAD ourselves?
- Train both models in one pipeline using CADLoss?
- use blensor to create new data?
